# Qwen2.5-VL vision-pathway causal validation on VLGuard

This A100 notebook builds a language-residual direction from disjoint VLGuard unsafe and safe **image groups** at layer 13, then evaluates baseline, repair, and equal-norm random-direction controls on held-out unsafe images. It is a causal refusal-ASR screen, not a human safety result and not a BLOCK-EM training result.

Prerequisite: one provenance-complete Qwen2.5-VL 3B candidate adapter from notebook `01q` **and its passed, provenance-bound face-sanity review**. VLGuard is gated; accept its research-use terms on Hugging Face before running. This notebook exports a strict direction package for the Step 3 cross-pathway comparison.

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
).strip()
print(gpu)
if 'A100' not in gpu:
    raise SystemExit('Select an A100 runtime; the frozen BF16 runtime rejects T4, L4, and TPU.')

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=False)
TRAINING_SEED = 42  # Qwen adapter seed; this is not the retired Gemma OOD seed.
if TRAINING_SEED not in (42, 43, 44):
    raise SystemExit('TRAINING_SEED must be 42, 43, or 44.')
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b').resolve()
EXPECTED_ROOT = Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b')
if DRIVE_PROJECT != EXPECTED_ROOT or not EXPECTED_ROOT.parent.is_dir():
    raise SystemExit('Expected the mounted, dedicated Qwen Drive root.')
VLGUARD_ROOT = DRIVE_PROJECT / 'data' / 'vlguard'
RUNS_DIR = DRIVE_PROJECT / 'runs'
ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_qwen2_5_vl_3b_faces_seed{TRAINING_SEED}'
REVIEW_SUMMARY = DRIVE_PROJECT / 'results' / f'review_qwen2_5_vl_3b_seed{TRAINING_SEED}_summary.json'
OUTPUT_DIR = DRIVE_PROJECT / 'results' / 'vlguard_vision' / f'seed{TRAINING_SEED}'
HF_HOME = DRIVE_PROJECT / 'cache' / 'huggingface'
for path in (RUNS_DIR, HF_HOME):
    path.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
print('Persistent Qwen root:', DRIVE_PROJECT)

In [ ]:
REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
REPO_REF = 'main'  # Replace with the recorded 40-character commit for the final run.
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise SystemExit(f'{REPO_DIR} is not a Git clone; restart Colab.')
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.rstrip('/') not in {REPO_URL.rstrip('/'), REPO_URL.removesuffix('.git')}:
        raise SystemExit(f'Unexpected origin {origin!r}; restart Colab.')
    dirty = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit('Existing runtime clone is dirty; restart Colab.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', '--tags', 'origin'])
else:
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)])
target = 'origin/main' if REPO_REF == 'main' else REPO_REF
REPO_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{target}^{{commit}}'], text=True
).strip()
subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT])
%cd {REPO_DIR}
print('Validation source commit:', REPO_COMMIT)

In [ ]:
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    raise SystemExit('Add HF_TOKEN to Colab secrets after accepting the VLGuard gate.')
os.environ['HF_TOKEN'] = token
from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print('Hugging Face session ready without writing Git credentials.')

## Build the exact A100 environment

This synchronizes the hash-locked Python 3.12 / CUDA 12.8 Qwen stack. The runner will independently reject a version, CUDA, BF16, or device mismatch.

In [ ]:
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
QWEN_ENV = Path('/content/qwen2-5-vl-a100-py312')
if not (QWEN_ENV / 'bin' / 'python').is_file():
    subprocess.check_call(['uv', 'venv', '--python', '3.12', str(QWEN_ENV)])
QWEN_PYTHON = QWEN_ENV / 'bin' / 'python'
subprocess.check_call([
    'uv', 'pip', 'sync', 'requirements/qwen-a100.lock',
    '--python', str(QWEN_PYTHON), '--torch-backend', 'cu128',
])

## Download, parse, and seal VLGuard roles

The official dataset is `train.json` plus `train.zip`, not a standard image-column `datasets` table. This cell pins the Hub revision, validates archive paths, hashes every selected image, and seals 100 safe direction images, 100 unsafe direction images, and 100 disjoint unsafe validation images.

In [ ]:
subprocess.check_call([
    str(QWEN_PYTHON), 'scripts/prepare_vlguard.py',
    '--root', str(VLGUARD_ROOT),
    '--direction-per-class', '100',
    '--validation-unsafe', '100',
])
MANIFEST_PATH = VLGUARD_ROOT / 'vlguard_vision_contrast_v1.json'
subprocess.check_call([
    str(QWEN_PYTHON), 'scripts/prepare_vlguard.py',
    '--root', str(VLGUARD_ROOT), '--validate-only',
])

In [ ]:
import yaml

RUN_CONFIG = RUNS_DIR / f'qwen_vlguard_vision_seed{TRAINING_SEED}.yaml'
source = yaml.safe_load(Path('configs/qwen_vlguard_vision_validation.yaml').read_text())
source.update({
    'adapter_dir': str(ADAPTER_DIR),
    'review_summary_path': str(REVIEW_SUMMARY),
    'manifest_path': str(MANIFEST_PATH),
    'image_root': str(VLGUARD_ROOT / 'images'),
    'output_dir': str(OUTPUT_DIR),
    'training_seed': TRAINING_SEED,
})
rendered = yaml.safe_dump(source, sort_keys=False)
if RUN_CONFIG.exists() and RUN_CONFIG.read_text() != rendered:
    raise SystemExit(f'Existing run config differs; archive the prior Qwen run: {RUN_CONFIG}')
if not RUN_CONFIG.exists():
    RUN_CONFIG.write_text(rendered)
subprocess.check_call([
    str(QWEN_PYTHON), 'scripts/validate_vlguard_vision.py',
    '--config', str(RUN_CONFIG), '--validate-config-only',
])

## Capture layer 13 and run baseline / repair / random controls

The direction uses a fixed neutral prompt. Validation uses each held-out unsafe image's original unsafe instruction. The pre-registered primary alpha is 150; 80 and 250 are sensitivity conditions. Image-token positions come from Qwen's processor output, never a fixed index. Generation rows are fsynced individually, so reconnecting and rerunning resumes the same immutable package.

In [ ]:
process = subprocess.Popen(
    [str(QWEN_PYTHON), '-u', 'scripts/validate_vlguard_vision.py', '--config', str(RUN_CONFIG)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env=os.environ.copy(),
)
with process:
    for line in process.stdout:
        print(line, end='')
if process.returncode:
    raise SystemExit(f'Vision validation failed with exit code {process.returncode}.')

In [ ]:
import json
summary_path = OUTPUT_DIR / 'summary.json'
if not summary_path.is_file():
    raise SystemExit('The immutable validation summary is missing.')
package_path = OUTPUT_DIR / 'direction_package.json'
if not package_path.is_file():
    raise SystemExit('The Step 3 vision-direction handoff package is missing.')
summary = json.loads(summary_path.read_text())
package = json.loads(package_path.read_text())
print(json.dumps({
    'status': summary['status'],
    'claim_boundary': summary['claim_boundary'],
    'commit': summary['commit'],
    'manifest_sha256': summary['manifest_sha256'],
    'primary_alpha': summary['primary_alpha'],
    'primary_comparison': summary['primary_comparison'],
    'generation_rows': summary['generation_rows'],
}, indent=2, sort_keys=True))
print('Step 3 vision package:', package_path)
print('Package fingerprint:', package['package_fingerprint'])
print('Keep generations.jsonl private: it contains unsafe prompts and model responses.')